# Lesson 30 · Softmax and cross-entropy — turning scores into beliefs, and deriving the gradient p − y by hand

**Goal**: turn the scores a model outputs (`logits`) into probabilities, then use cross-entropy to compute the loss for the correct class.

**Why it matters for Aurora**: the output layers of the Month 2 keyword classifier and the Month 3 ASR model both call `softmax`, and the `cross_entropy` in the training loop is the ruler that steers every gradient update.


← **Previous lesson** · [L29 · Common distributions](L29_distributions.ipynb)

> Last lesson covered **common distributions**: uniform, Gaussian, Bernoulli — PDF / CDF and sampling.
> This lesson explores **softmax and cross-entropy**.

## Learning goals

By the end of this lesson you should be able to:

1. Implement a numerically stable `softmax(z)` that turns any logits vector into a probability distribution summing to 1
2. Implement `cross_entropy(probs, true_idx)` = `-log(probs[true_idx])`, and understand why it only looks at the true class's slot
3. Explain what each term in H(p,q) = H(p) + D_KL(p‖q) means, and the difference between `log₂` (bits) and `ln` (nats)
4. Derive the gradient formula `dL/dz_i = p_i − y_i` by hand and check it numerically


## This lesson's story: the model gives us scores, but we want probabilities

You've trained a keyword spotter, and its last layer outputs three numbers:

```
logits = [2.0, 1.0, 0.1]  ← for speech / music / noise
```

These numbers are not probabilities — they can be negative, they don't add up to 1, and they have no clear meaning on their own.

**softmax** turns logits into a probability distribution:

$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Result: [0.659, 0.242, 0.099] — sums to 1, and every value lies in (0,1).

**Cross-entropy (CE)** measures how far this probability distribution is from the "ideal answer":

$$\text{CE} = -\log(p_{\text{true}})$$

If the true class is speech (idx=0), then CE = -log(0.659) ≈ 0.417.
The more confident the model (p[true]→1), the smaller the loss (-log(1)=0).

Why do these two functions matter so much?
The last two lines of almost every classification network are: `probs = softmax(logits)` + `loss = cross_entropy(probs, target)`.
In this lesson you derive the gradient by hand: `dL/dz_i = p_i - y_i` (softmax probabilities minus the one-hot label) — the formula backpropagation starts from at the last layer.

## 🤔 Why did engineers invent this?

- **What happens without it?** The model's last layer spits out arbitrary real numbers like `[2.0, 1.0, 0.1]` (logits). They can be negative and don't sum to 1, so you can't read them directly as "how sure am I", and you can't compare them with a label to compute a loss.
- **What real problem does it solve?** softmax squashes that row of scores into a valid probability distribution (every entry ∈ (0,1), summing to 1); cross-entropy then turns "how far the predicted distribution is from the right answer" into a **differentiable scalar loss**, which gives training a gradient to follow.
- **Where does it come back later?** It's the last two lines of almost every classification network — keyword spotting, **L63** (the KWS model), ASR decoding, and an LLM predicting the next token.

## ⚠️ Common misconceptions

> Don't think of softmax as "a little trick for normalizing numbers". What it really does is turn logits into a **differentiable probability distribution** — and precisely because it is smooth and differentiable, the cross-entropy loss can flow all the way back to every logit. That's the `dL/dz_i = p_i − y_i` you'll derive by hand at the end of this lesson.

## 1. logits: the model gives scores first, not probabilities

A classifier's last layer usually outputs a set of arbitrary real numbers called `logits`. They can be negative, and they don't need to add up to 1. softmax's job is to turn this set of scores into a probability distribution.

**Why softmax subtracts the maximum first**: `np.exp(z)` overflows to `inf` when z > 709. After subtracting `max(z)`, the largest exponential becomes `exp(0) = 1` and the rest are smaller, so nothing overflows. The change doesn't affect the result — numerator and denominator are both divided by `exp(max(z))`, so the probabilities are unchanged.

**Why cross_entropy only looks at the true class**: the training goal is to maximize the probability the model assigns to the correct class, i.e. maximize `log P(correct)`, which is the same as minimizing `-log P(correct)`. As for the other classes' probabilities, softmax's normalization already pushes them down along the way, so they don't need their own terms in the loss.

### 🔍 Why does cross-entropy only look at the true class? — the maximum-likelihood view

You might ask: "If I want the model to be more confident, why not put **every** class's probability into the loss — say, penalize high probabilities on the wrong classes?"

**Answer**: softmax's constraint (all probabilities sum to 1) already takes care of that.

Picture the probabilistic model of a classification problem: given a piece of audio x whose true class is t (say, speech), the model's probability distribution is `q(y|x) = softmax(z)`.

The principle of maximum likelihood says: the best parameters are the ones that **maximize the probability of the true label**, i.e. maximize `log q(t|x)`.

```
maximize log(q_t)  ≡  minimize -log(q_t)
     ↑                    ↑
high true-class prob   cross-entropy loss (only the q_t term)
```

Because softmax has the constraint `Σ q_y = 1`:
- when you maximize `q_t` (the correct class's probability)
- the other classes' probabilities are automatically pushed down by softmax's "pie" constraint — because the total is fixed at 1

This is a natural property of a conditional probability distribution and **doesn't need to be written into the loss separately**. Formally, it is equivalent to maximum-likelihood estimation against the true distribution `p(y|x) = [0, ..., 1, ..., 0]` (one-hot).

### 📌 One-hot encoding: the standard form of a label

In cross-entropy and the gradient derivation, the "true label" is represented as a **one-hot vector**:

```
the true class is class t (say t=0)
    ↓
one-hot: y = [1, 0, 0, ...]
            ↑ 1 at the true class's position
            0 everywhere else
```

**Why encode it this way?**
- Mathematically, the one-hot y has the same format as the probability distribution p (from softmax): both are vectors with non-negative entries summing to 1
- Cross-entropy can be written as `H(p, q) = -Σ_i y_i log(q_i)`; since y contains a single 1, only the true class's term contributes
- It lets the gradient formula `dL/dz = p - y` be written in one piece (derived later)

A common one-hot conversion in code:
```python
true_idx = 0           # index of the true class
y = np.zeros(3)        # a three-class problem
y[true_idx] = 1.0      # set that position to 1
# result: y = [1.0, 0.0, 0.0]
```

In [ ]:
import numpy as np

logits = np.array([2.0, 1.0, 0.1])
print('logits =', logits)
print('sum(logits) =', logits.sum(), ' not a probability sum')
print('argmax =', int(np.argmax(logits)), ' the class with the highest score')


## 2. A numerically stable softmax

**Why does it need to be "numerically stable"?**

If you compute `exp(z)` directly and z contains a large value (say z=1000), `exp(1000)` overflows to `inf`.

**A mathematically equivalent trick**: first subtract the maximum c = max(z), then take exp:

$$\text{softmax}(z)_i = \frac{e^{z_i - c}}{\sum_j e^{z_j - c}}$$

This doesn't change the result (numerator and denominator are both multiplied by e^{-c}), but it turns the largest exponent into 0 and so avoids overflow.

**Hand check** (z = [2, 1, 0.1], c = 2):

```
exp([0, -1, -1.9]) = [1.000, 0.368, 0.150]
sum = 1.518
softmax = [0.659, 0.242, 0.099]   ← sum = 1 ✓
```

**Key properties**:
- If all logits are equal, softmax outputs a uniform distribution: [1/n, 1/n, ..., 1/n]
- The bigger the gaps between logits, the more concentrated the probabilities (the top class goes to 1, the rest to 0)
- Gradient: `dL/dz_i = p_i - y_i` (p=softmax, y=one-hot)

> **Implementation hint**: `z_stable = z - z.max(); exp_z = np.exp(z_stable); return exp_z / exp_z.sum()`

### 💭 Why does softmax's "pie" push down the wrong classes automatically? — a property of the exponential

A student might ask: if I only optimize "the probability of the correct class", what guarantees that the other classes' probabilities go down?

**The answer comes from one key property of the exponential**: **exp() magnifies differences**.

**A concrete example**:
- logits = [2.0, 1.0, 0.1] (three classes)
- Without exp(), with a plain normalization (dividing by the total): [2.0, 1.0, 0.1] / 3.1 = [0.645, 0.323, 0.032]
- With softmax (exp first, then normalize):
  - exp([2.0, 1.0, 0.1]) = [7.39, 2.72, 1.11]
  - softmax = [0.659, 0.242, 0.099]

Not much different? Look at another example where the scores are **further apart**:

| logits | Plain division | Softmax probabilities |
|--------|---------|------------|
| [5.0, 1.0, 0.1] | [0.820, 0.164, 0.016] | [0.975, 0.018, 0.007] |
| [10.0, 1.0, 0.1] | [0.901, 0.090, 0.009] | [0.9998, 0.0001, 0.00005] |

**Key observation**: exp() magnifies score differences **non-linearly**. The highest score gets an even higher probability, the lowest an even lower one. That's why optimizing only `p_t` (the correct class) during training is enough — the exponential form of softmax already "penalizes" the wrong classes automatically.

**Mathematical intuition**: softmax's numerator is $e^{z_i}$ and its denominator is $\sum_j e^{z_j}$. When you increase $z_t$ (the correct class's score):
- the numerator $e^{z_t}$ grows exponentially
- the other $e^{z_j}$ ($j \neq t$) make up a smaller share of the denominator
- the constraint $\sum p = 1$ automatically turns that growth into lower probabilities for the other classes



## Into the lab: how scores turn into probabilities

Fix a set of logits and watch how the probability vector softmax outputs becomes more concentrated as the gaps between logits widen; then fix the target and see how the value of `cross_entropy` moves in the opposite direction to the correct class's probability.


### Before writing code, fill in the variable table

Before writing `softmax`, pin down three things:
- Input: `z`, a vector of arbitrary real numbers (logits), shape `(n_classes,)`
- Key steps: subtract `max(z)` to prevent overflow → normalize the result of `exp(z)`
- Return: a probability vector, every entry ≥ 0, summing to exactly 1


In [ ]:
import numpy as np
def softmax(z):
    z = np.asarray(z, dtype=float)
    # ✏️ TODO: return a probability distribution (summing to 1)
    raise NotImplementedError("Implement softmax: subtract max(z), normalize exp(z), and return the probability vector")


### 🔧 NumPy quick reference: the functions used here

**`np.asarray(z, dtype=float)`**:
- What it does: turns the input into a NumPy array and makes sure the data type is floating point
- Why do we need it? If the input is a list, a tuple or integers, calling `np.exp()` on it directly can cause type trouble. `dtype=float` makes sure the later log and division operations don't lose precision.
- Example: `np.asarray([2, 1, 0.1], dtype=float)` → `array([2. , 1. , 0.1])`

**`np.log(x)` vs `np.log2(x)`**:
- `np.log(x)`: the natural logarithm, ln(x) (base e ≈ 2.718)
- `np.log2(x)`: the base-2 logarithm (units: bits)
- Which one does the code use? In this lesson, the cross-entropy loss usually uses `np.log()` (natural log); if you want Shannon entropy in bits, change the base: `np.log(p) / np.log(2)` is equivalent to `np.log2(p)`



## Hands-on: how score gaps change the probabilities

Keep the order of the classes the same and only stretch the gaps between the logits. The bigger the gaps, the more concentrated the probabilities after softmax.


In [ ]:
import numpy as np

base = np.array([2.0, 1.0, 0.1])
for scale in [0.5, 1.0, 2.0, 5.0]:
    z = base * scale
    e = np.exp(z - np.max(z))
    p = e / e.sum()
    print(f'scale={scale:>3}: logits={np.round(z,2)} -> probs={np.round(p,3)}')


## Code experiment: same correct class, different confidence

Cross-entropy only reads the probability in the true class's slot. The higher the correct class's probability, the smaller the loss.


In [ ]:
try:
    # finish softmax(z) above before running this
    for z in [[2.0, 1.0, 0.1], [5.0, 1.0, 0.1], [1.1, 1.0, 0.9]]:
        p = softmax(z)
        print('logits=', z, 'probs=', np.round(p, 3), 'true-class prob=', round(float(p[0]), 3))
except (NotImplementedError, TypeError) as _e:
    print('⬜ Implement the function first, then run this cell:', _e)


In [ ]:
p = softmax([2.0, 1.0, 0.1])
print('probabilities:', np.round(p, 3), ' sum =', round(float(p.sum()), 6))
assert abs(p.sum() - 1.0) < 1e-9, 'the probabilities should sum to 1'
assert np.argmax(p) == 0, 'the highest score should get the highest probability'
print('✅ softmax passed.')

## 3. Shannon entropy — a measure of uncertainty

**Definition**: for a discrete probability distribution p = (p₁, ..., pₙ), the Shannon entropy is

> H(p) = −Σᵢ pᵢ log₂ pᵢ

(by convention 0·log 0 = 0; with log₂ the unit is the bit, with ln it is the nat)

**Three key properties**:

| Property | Expression | Intuition |
|------|--------|------|
| Non-negativity | H(p) ≥ 0 | information is never negative |
| Maximum | H(uniform) = log₂ n | uncertainty is highest when you know nothing |
| Certainty | H(certain outcome) = 0 | when you already know the result, you need no information |

**Relationship to cross-entropy**:

```
H(p, q) = H(p) + D_KL(p‖q)
cross-entropy = entropy + KL divergence (the distance from the true distribution p to the predicted distribution q)
```

**Why training a classifier with cross-entropy is the same as minimizing KL divergence**:
the true labels p are fixed over the training set → H(p) is a constant → minimizing H(p,q) = minimizing D_KL(p‖q).
That's the mathematical reason the cross_entropy loss can drive the model's predicted distribution toward the true one.

**Aurora connection**: the more uncertain the true labels of audio events are (the larger H(p)), the less "useful information" the same batch of data provides, and the more samples the model needs to converge.

### 📊 KL divergence: the "distance" between two probability distributions

To see exactly why "cross-entropy is equivalent to minimizing KL divergence", we need to understand **KL divergence** (Kullback–Leibler divergence).

**Definition**: given two probability distributions p and q (over the same sample space), the KL divergence is

$$D_{\text{KL}}(p \| q) = \sum_i p_i \log \frac{p_i}{q_i} = \sum_i p_i \log(p_i) - \sum_i p_i \log(q_i)$$

**Intuition**: KL divergence measures the cost of using distribution q to "approximate" distribution p. It isn't symmetric — `D_KL(p||q)` and `D_KL(q||p)` are usually different.

**Three key properties**:

| Property | Expression | Meaning |
|------|--------|------|
| Non-negativity | D_KL(p\|\|q) ≥ 0 | the distance is never negative |
| Equality | D_KL(p\|\|q) = 0 ⟺ p = q | the distance is zero when the two distributions are the same |
| Asymptotic behavior | q_i → 0 but p_i ≠ 0 ⟹ D_KL → ∞ | if q thinks an event is impossible but p thinks it can happen, the distance is infinite |

**How KL divergence relates to cross-entropy**:

Look at the expanded KL divergence:

$$D_{\text{KL}}(p \| q) = \sum_i p_i \log(p_i) - \sum_i p_i \log(q_i)$$

The first term `Σ p_i log(p_i)` is **minus the entropy, −H(p)** (definition above); the second term `-Σ p_i log(q_i)` is exactly the **cross-entropy H(p, q)**. So D_KL(p‖q) = −H(p) + H(p, q), that is:

$$H(p, q) = H(p) + D_{\text{KL}}(p \| q)$$

**Why does this matter so much for training a classifier?**
- The true label distribution p (one-hot, e.g. [1,0,0]) is **fixed** over the training set
- So H(p) is a constant
- Minimizing cross-entropy ≡ minimizing D_KL(p||q)
- In other words: training is, at heart, about moving the model's predicted distribution q step by step toward the true distribution p



In [ ]:
import numpy as np

def shannon_entropy(p, base=2):
    """
    H(p) = -Σ p_i log_b(p_i), with the convention 0·log(0) = 0

    Parameters:
      p: probability distribution (a vector summing to 1)
      base: base of the logarithm. base=2 gives bits; base=e (≈2.718) gives nats

    Key steps:
      1. p = p[p > 0]: skip zero-probability terms (they contribute 0 to the entropy)
      2. np.log(p): natural logarithm (ln)
      3. np.log(base): change of base — log_b(p_i) = ln(p_i) / ln(b)
    """
    p = np.asarray(p, dtype=float)
    p = p[p > 0]          # skip the p_i=0 terms (0·log0 → 0)
    # change-of-base formula: log_b(x) = ln(x) / ln(b)
    # np.log is the natural log ln; np.log2(p) would give log₂(p) directly
    return float(-np.sum(p * np.log(p)) / np.log(base))

# ── Properties 1 & 3: the uniform distribution has the most entropy; a certain one has 0 ──
n       = 4
uniform = np.ones(n) / n
peaked  = np.array([0.97, 0.01, 0.01, 0.01])
certain = np.array([1.0, 0.0, 0.0, 0.0])

print('Entropy (bits) compared:')
print(f'  uniform    H = {shannon_entropy(uniform):.4f}  (theoretical maximum log₂{n} = {np.log2(n):.4f})')
print(f'  peaked     H = {shannon_entropy(peaked):.4f}')
print(f'  certain    H = {shannon_entropy(certain):.4f}  ← no uncertainty')

# ── Property 2: H(p,q) ≥ H(p), with equality if and only if p=q ──────────────
def cross_entropy_h(p, q):
    """H(p,q) = -Σ p_i log₂ q_i (base 2, in bits)"""
    p, q = np.asarray(p, float), np.clip(np.asarray(q, float), 1e-15, None)
    return float(-np.sum(p * np.log(q)) / np.log(2))

p_true = np.array([0.7, 0.2, 0.1])
q_good = np.array([0.65, 0.25, 0.10])   # close to the truth
q_bad  = np.array([0.10, 0.10, 0.80])   # far from the truth

Hp       = shannon_entropy(p_true)
Hpq_good = cross_entropy_h(p_true, q_good)
Hpq_bad  = cross_entropy_h(p_true, q_bad)

print(f'\nentropy H(p)        = {Hp:.4f} bits  ← the theoretical lower bound of the loss (reached only when D_KL=0)')
print(f'H(p, q_good)  = {Hpq_good:.4f} bits  KL = {Hpq_good - Hp:.4f}')
print(f'H(p, q_bad)   = {Hpq_bad:.4f} bits  KL = {Hpq_bad - Hp:.4f}')
print('→ the training goal is to squeeze KL; since H(p) is fixed, minimizing cross-entropy ≡ minimizing KL divergence')

assert shannon_entropy(certain) < 1e-9,          'a certain distribution should have entropy 0'
assert abs(shannon_entropy(uniform) - np.log2(n)) < 1e-9, 'a uniform distribution should have entropy log₂(n)'
assert Hpq_good >= Hp - 1e-9, 'lower-bound property H(p,q) ≥ H(p) failed'
assert Hpq_bad  >= Hp - 1e-9, 'lower-bound property H(p,q) ≥ H(p) failed'
print('✅ All three Shannon-entropy properties verified')

### 💡 Why does Shannon entropy look like this? — a story from information theory

Here's a puzzle: **how many yes/no questions do you need to guess the outcome of a probability distribution**?

For example, for a fair coin (p=[0.5, 0.5]) you need **1** yes/no question to know whether it's heads or tails.
For a four-sided die (p=[0.25, 0.25, 0.25, 0.25]) you need **2** yes/no questions.
But if the die is heavily loaded (p=[0.99, 0.01/3, 0.01/3, 0.01/3]), the result is nearly always the first face, and **0 questions** are pretty much enough.

**Shannon's 1948 insight**: given a distribution p, the average number of "yes/no questions" needed is

$$H(p) = \sum_i p_i \log_2 \frac{1}{p_i} = -\sum_i p_i \log_2 p_i \quad \text{(unit: bits)}$$

That is Shannon entropy. Why `-log p_i`? Because **the surprise of an outcome grows with 1/p_i**:
- p_i large (close to 1) → `log(1/p_i)` small → the event tells you nothing new
- p_i small (close to 0) → `log(1/p_i)` large → when the event happens, it's a big surprise

Take the expectation over the whole distribution (weighting by p_i) and you get the **average information**: Shannon entropy.

**The mathematical reason for the convention "0·log 0 = 0"**:

For an event with probability 0, even though `log(1/p) = ∞`, because p=0 its contribution to the expectation Σ p_i log(1/p_i) is:

$$\lim_{p \to 0^+} p \log(1/p) = \lim_{p \to 0^+} -p \log(p) = 0$$

(provable with L'Hôpital's rule)

So in code we can simply skip the p_i=0 terms; they contribute 0.

**The three properties of Shannon entropy** (now with the information-theory background):

| Property | Expression | Meaning |
|------|--------|------|
| Non-negativity | H(p) ≥ 0 | information is never negative |
| Maximum | H(uniform) = log₂ n | when you have no idea what will happen, uncertainty is at its peak |
| Certainty | H(certain outcome) = 0 | when you already know the result, you need no new information |



## 4. ✏️ Implement cross-entropy `cross_entropy(probs, true_idx)`

Look only at the true class's probability: `loss = −log(probs[true_idx])`. The more confident and correct the prediction, the smaller the loss.

**Line of reasoning**:
1. `probs` is the output of softmax: every entry ≥ 0, summing to 1; `true_idx` is the index of the correct class.
2. Use `probs[true_idx]` to pick out the correct class's probability p. The closer p is to 1, the more confident and correct the model.
3. Take `-log(p)`: when p=1 the loss is 0 (perfect), when p=0.1 the loss is about 2.3, and as p→0 the loss→∞. One line of code is enough.

**Reference input and output**: `probs=[0.665, 0.245, 0.090]`, `true_idx=0` → `loss = -log(0.665) ≈ 0.408`; with `true_idx=2` → `loss = -log(0.090) ≈ 2.408`.

> Note: `log` here means the natural logarithm `ln`; the Shannon-entropy section above used `log₂`. The two differ only by a constant factor of `ln(2)`.

> 💡 Need a hint? Once you've done the exercise, you can compare with the reference implementation in the `solutions/` folder.


In [ ]:
def cross_entropy(probs, true_idx):
    # ✏️ TODO: return the cross-entropy loss
    raise NotImplementedError("Implement cross_entropy: return -np.log(probs[true_idx])")


In [ ]:
try:
    p = softmax([2.0, 1.0, 0.1])
    good = cross_entropy(p, 0)   # true class = the highest-scoring one -> small loss
    bad  = cross_entropy(p, 2)   # true class = the lowest-scoring one -> large loss
    print('loss when right:', round(good,3), ' loss when wrong:', round(bad,3))
    assert good < bad
    print('✅ Passed: you have built the loss function for classification training.')
except (NotImplementedError, TypeError) as _e:
    print('⬜ Implement the function first, then run this cell:', _e)


## 5. Toy classifier: from a row of scores to a single loss

Below we run one sample's three-class output all the way through. The true class is `target=0`, so the loss only looks at the probability of class 0 after softmax.


In [ ]:
try:
    class_names = ['speech', 'music', 'noise']
    logits = np.array([3.2, 1.4, -0.5])
    target = 0

    probs = softmax(logits)
    loss = cross_entropy(probs, target)

    for name, z, p in zip(class_names, logits, probs):
        print(f'{name:>6}: logit={z:+.2f}  prob={p:.3f}')
    print('target =', class_names[target], 'loss =', round(float(loss), 3))
except (NotImplementedError, TypeError) as _e:
    print('⬜ Implement the function first, then run this cell:', _e)


**🔗 Aurora connection**: the output layer of the Month 2 keyword classifier (target path: `src/aurora/speech/`, to be built in Month 2) calls `softmax(z)` directly, and `cross_entropy(probs, target)` in the training loop produces the scalar loss that the gradient flows back from. Month 3's ASR repeats the same process frame by frame at the token level, with a vocabulary that can run to thousands of classes, but the logic of `softmax` and `cross_entropy` is exactly the same as here.

In [ ]:
try:
    for scale in [0.5, 1.0, 2.0, 5.0]:
        z = np.array([2.0, 1.0, 0.1]) * scale
        p = softmax(z)
        loss = cross_entropy(p, 0)
        print(f'scale={scale:>3}: p_true={p[0]:.3f}, loss={loss:.3f}')
except (NotImplementedError, TypeError) as _e:
    print('⬜ Implement the function first, then run this cell:', _e)


## Parameter experiment: turn one knob at a time

Raise the logit for `true_idx=0` step by step from 2 to 10 and watch the softmax probability climb from 0.659 toward 1.0 while cross_entropy falls from 0.417 toward 0 — the higher the correct class's score, the more confident the model and the lower the loss.

Then add 100 to every logit at once (`z + 100`) and confirm that the softmax output is exactly the same as before. That's the whole point of "subtracting the max": absolute values don't affect the probability distribution; only the relative gaps between classes decide the probabilities.

## 6. Deriving the gradient by hand: dL/dz_i = p_i - y_i

Let `z` be the logits, `p = softmax(z)`, and `y` the one-hot label (1 at the true class t, 0 elsewhere).
Cross-entropy loss: `L = -log(p_t)`.

**Derivation** (partial derivative with respect to logit z_i):

```
L = -log(p_t) = -log(exp(z_t) / sum_j exp(z_j))
  = -z_t + log(sum_j exp(z_j))

dL/dz_i = d/dz_i [-z_t + log(sum_j exp(z_j))]

if i = t:  dL/dz_t = -1 + exp(z_t)/sum = -1 + p_t = p_t - 1 = p_t - y_t
if i != t: dL/dz_i =  0 + exp(z_i)/sum =      p_i = p_i - 0 = p_i - y_i

in one line: dL/dz_i = p_i - y_i
```

The intuition behind this gradient: **the difference between the predicted probability and the true label** is the gradient of the logit.
The closer the prediction is to the true label (`p_t → 1`), the closer the gradient `p_t - 1` gets to 0, and the smaller the update.

**Aurora connection**: the `backward()` of the Month 2 keyword classifier starts from `cross_entropy`, and the very first thing it computes is `p - y`, which is passed into backpropagation through the last layer.

In [ ]:
# Reference implementation (_softmax_ref): only for checking the gradient; it doesn't touch the softmax you implemented above
import numpy as np

def _softmax_ref(z):  # reference implementation, doesn't clash with the student's softmax
    z = np.asarray(z, dtype=float)
    e = np.exp(z - z.max())
    return e / e.sum()

def ce_gradient(z, true_idx):
    p = _softmax_ref(z)
    y = np.zeros_like(p)
    y[true_idx] = 1.0
    return p - y  # dL/dz = p - y

# Check: numerical gradient vs analytic gradient
def cross_entropy_loss(z, true_idx):
    p = _softmax_ref(z)
    return -np.log(_softmax_ref(z)[true_idx] + 1e-15)

z = np.array([3.2, 1.4, -0.5])
true_idx = 0
h = 1e-5

analytic_grad = ce_gradient(z, true_idx)
numerical_grad = np.array([
    (cross_entropy_loss(z + h*(np.arange(len(z))==i), true_idx) -
     cross_entropy_loss(z - h*(np.arange(len(z))==i), true_idx)) / (2*h)
    for i in range(len(z))
])

print('analytic gradient (p - y):', np.round(analytic_grad, 6))
print('numerical gradient:       ', np.round(numerical_grad, 6))
print('error:                    ', np.round(np.abs(analytic_grad - numerical_grad), 8))

assert np.allclose(analytic_grad, numerical_grad, atol=1e-6)
print('✅ dL/dz_i = p_i - y_i verified: the analytic and numerical gradients agree')


### 📐 Step-by-step derivation: d/dz_i log(Σ_j exp(z_j))

The gradient derivation involves the chain rule and partial derivatives of a fairly involved expression. Below we unpack it step by step — ideal if your calculus background is from high school.

**Step 0**: the original form of the cross-entropy loss
$$L = -\log(p_t) = -\log\left(\frac{e^{z_t}}{\sum_j e^{z_j}}\right)$$

where $z_t$ is the true class's logit and $\sum_j e^{z_j}$ is the sum of the exponentials of all logits.

**Step 1**: apply the division rule for logarithms, $\log(a/b) = \log(a) - \log(b)$
$$L = -\left[\log(e^{z_t}) - \log\left(\sum_j e^{z_j}\right)\right] = -z_t + \log\left(\sum_j e^{z_j}\right)$$

**Step 2**: take the partial derivative with respect to one logit $z_i$

Now compute $\frac{\partial L}{\partial z_i}$. The partial derivative of the first term $-z_t$:
- if $i = t$ (the correct class), then $\frac{\partial (-z_t)}{\partial z_i} = -1$
- if $i \neq t$ (a wrong class), then $\frac{\partial (-z_t)}{\partial z_i} = 0$

The partial derivative of the second term $\log(\sum_j e^{z_j})$ uses the **chain rule**:

$$\frac{\partial}{\partial z_i} \log\left(\sum_j e^{z_j}\right) = \frac{1}{\sum_j e^{z_j}} \cdot \frac{\partial}{\partial z_i}\left(\sum_j e^{z_j}\right)$$

Differentiating the sum (only the $j=i$ term is non-zero):
$$\frac{\partial}{\partial z_i}\left(\sum_j e^{z_j}\right) = e^{z_i}$$

So:
$$\frac{\partial}{\partial z_i} \log\left(\sum_j e^{z_j}\right) = \frac{e^{z_i}}{\sum_j e^{z_j}} = p_i$$

Here $p_i$ is just the i-th component of the softmax.

**Step 3**: combine the two terms
$$\frac{\partial L}{\partial z_i} = -\mathbb{1}_{i=t} + p_i = p_i - \mathbb{1}_{i=t}$$

where $\mathbb{1}_{i=t}$ is the indicator function (1 when $i=t$, 0 otherwise).

**Step 4**: rewrite with the one-hot vector

Call the one-hot label vector $y$, with $y_i = \mathbb{1}_{i=t}$ (1 at the true class's position, 0 elsewhere). Then:

$$\boxed{\frac{\partial L}{\partial z_i} = p_i - y_i}$$

That's the gradient formula. **Intuition**:
- When the prediction is right ($p_t \approx 1$), the gradient $p_t - y_t = 1 - 1 = 0$: no update needed
- When the prediction is wrong ($p_t \ll 1$), the gradient is large in magnitude and the parameters get a big adjustment



## Wrap-up

You can now use `softmax(z)` to turn any logits vector into a probability distribution, and `cross_entropy(probs, target)` to get a scalar loss. Together, the two functions are a classifier's complete "scores → probabilities → loss" pipeline. The Month 2 keyword classifier will reuse this code directly, and the Month 3 ASR computes its training loss the same way at the token level.

**Next lesson L31** draws softmax, cross-entropy and Shannon entropy as pictures; after it, and before moving on to Audio DSP (L32), it's a good idea to work through the module-switch checklist at the end of L31.

The cross-entropy gradient won't stay a formula on paper: the training loop in L58 wires `dL/dz = p − y` into real parameter updates, and further on, sequence losses (such as CTC in L69) are just another view of the same idea.

In [ ]:
try:
    # Quick check: the maths of softmax + cross-entropy
    import numpy as np

    # Pattern 1: the bigger the logit gaps, the more concentrated the top probability
    for scale in [0.1, 1.0, 5.0, 20.0]:
        z = np.array([2.0, 1.0, 0.0]) * scale
        p = softmax(z)
        loss = cross_entropy(p, 0)  # target class = 0 (the top score)
        if p is None or loss is None:
            print('⬜ Implement softmax and cross_entropy first')
            break
        print(f'scale={scale:5.1f}  max_prob={p[0]:.4f}  loss={loss:.4f}')
    print('→ the larger the scale, the more confident the model and the smaller the cross-entropy loss.')
except (NotImplementedError, TypeError) as _e:
    print('⬜ Implement the function first, then run this cell:', _e)


---
⬇️ **Pass check**: you've read the wrap-up; finish the whiteboard challenge below before ticking off the self-review.


## ✏️ Whiteboard challenge: softmax + cross-entropy by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: z = [2.0, 1.0, 0.1]; compute the numerically stable softmax by hand:
- c = max(z) = ?
- exp(z - c) = ? (three values)
- softmax(z) = ? (three values; they should sum to 1)

**Question 2**: if the true class is class 0 (speech), cross_entropy = -log(p[0]) = ?

**Question 3**: if z = [5, 5, 5] (three equal scores), what does softmax output?
The cross-entropy is -log(?) ≈ ?

**Question 4**: the gradient of softmax + cross-entropy with respect to the logits: `dL/dz_i = ?`
(Hint: one-hot label y, softmax probabilities p)

When you've worked them out, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: softmax([2,1,0.1])
z = np.array([2.0, 1.0, 0.1])
c = z.max()   # = 2.0
exp_z = np.exp(z - c)
sm_expected = exp_z / exp_z.sum()
assert np.isclose(sm_expected.sum(), 1.0, atol=1e-12)
assert np.isclose(sm_expected[0], 0.6590011, atol=1e-5)
try:
    sm_computed = softmax(z)
    assert np.allclose(sm_computed, sm_expected, atol=1e-10)
    print(f"Q1 ✅  softmax([2,1,0.1])={np.round(sm_computed,4)}, sum={sm_computed.sum():.6f}")
except (NotImplementedError, TypeError):
    print("⬜ Q1: implement softmax() first, then run the answer-check cell")

# Q2: CE = -log(p[0]) ≈ 0.4170
ce_expected = -np.log(sm_expected[0])
assert np.isclose(ce_expected, 0.41703, atol=1e-4)
try:
    ce_computed = cross_entropy(sm_expected, 0)
    assert np.isclose(ce_computed, ce_expected, atol=1e-10)
    print(f"Q2 ✅  CE=-log({sm_expected[0]:.4f})={ce_computed:.5f}")
except (NotImplementedError, TypeError):
    print("⬜ Q2: implement cross_entropy() first, then run the answer-check cell")

# Q3: z=[5,5,5] → softmax=[1/3,1/3,1/3]
z3 = np.array([5.0, 5.0, 5.0])
sm3 = np.exp(z3 - z3.max()) / np.exp(z3 - z3.max()).sum()
assert np.allclose(sm3, [1/3, 1/3, 1/3], atol=1e-12)
ce3 = -np.log(1/3)
print(f"Q3 ✅  z=[5,5,5]→softmax=[1/3,1/3,1/3], CE=-log(1/3)={ce3:.4f}≈1.099")

# Q4: gradient dL/dz_i = p_i - y_i (one-hot: y[0]=1, rest=0)
y = np.array([1.0, 0.0, 0.0])
grad_expected = sm_expected - y
assert np.isclose(grad_expected.sum(), 0.0, atol=1e-12)  # the gradient sums to 0
print(f"Q4 ✅  dL/dz = p - y = {np.round(grad_expected, 4)}  (sums to 0, as it should)")
print(f"     i.e. dL/dz_0=p[0]-1 (the classifier 'pushes up' the correct class's probability)")
print("\n🎉 Softmax + cross-entropy whiteboard challenge passed! The classifier output-layer formulas are now second nature.")

In [ ]:
# ✏️ Self-review for this lesson
l30_review = {
    "softmax_formula_stable":     None,  # remember the numerically stable softmax (subtract the max)? True/False
    "softmax_implemented":        None,  # softmax implemented and assertions pass? True/False
    "cross_entropy_implemented":  None,  # cross_entropy implemented and assertions pass? True/False
    "gradient_formula":           None,  # remember dL/dz_i = p_i - y_i? True/False
    "whiteboard_passed":          None,  # whiteboard challenge done on paper? True/False
}

unfilled = [k for k, v in l30_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l30_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L30 fully cleared! On to L31: Distributions visualized')

---

→ **Next lesson** · [L31 · Distributions visualized](L31_visual_probability.ipynb)

> Next lesson covers **distributions visualized**: PDF, CDF and the cross-entropy loss surface, shown in pictures.